In [1]:
!pip install langchain langchain-openai langchain-community langgraph python-dotenv faiss-cpu pypdf

   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.2 MB 5.6 MB/s eta 0:00:03
   ---- ----------------------------------- 1.8/16.2 MB 4.4 MB/s eta 0:00:04
   ------ --------------------------------- 2.6/16.2 MB 4.2 MB/s eta 0:00:04
   -------- ------------------------------- 3.4/16.2 MB 4.1 MB/s eta 0:00:04
   ---------- ----------------------------- 4.2/16.2 MB 4.1 MB/s eta 0:00:03
   ------------ --------------------------- 5.0/16.2 MB 4.1 MB/s eta 0:00:03
   -------------- ------------------------- 5.8/16.2 MB 4.0 MB/s eta 0:00:03
   ---------------- ----------------------- 6.6/16.2 MB 4.0 MB/s eta 0:00:03
   ------------------ --------------------- 7.3/16.2 MB 4.0 MB/s eta 0:00:03
   -------------------- ------------------- 8.1/16.2 MB 4.0 MB/s eta 0:00:03
   --------------------- ------------------ 8.9/16.2 MB 4.0 MB/s eta 0:00:02
   ----------------------- ---------------- 9.7/16.2 MB 4.0 MB/s eta 0:00:02
   ---

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [5]:
load_dotenv()

False

In [ ]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [ ]:
loader = PyPDFLoader("intro-to-ml.pdf")
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
splitter = RecursiveCharacterSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store = vector_store.as_retreiver(search_type = 'similarity', search_kwargs={'k':4})

In [7]:
@tool
def rag_tool(query):
    """
    retrieve relevant information from the pdf document.
    use this tool when the user asks about factual / conceptual questions
    that might be answered from the stored documents.
    """

    result = retreiver.invoke(query)
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
    'query': query,
    'context' : context,
    'metadata' : metadata }

In [ ]:
tools = [rag_tool]
llm_with_tool = llm.bind_tools(tools)

In [10]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [11]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edge('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()

In [ ]:
chatbot

In [ ]:
result = chatbot.invoke(
    {
        'messages': [
            HumanMessage(
                content = (
                    "using the pdf notes, explain how to find the ideal value of k in the KNN"
                )
            )
        ]
    }
)

In [ ]:
print(result['messages'][:-1].content)